# scipy.optimize / interpolate / signal cheat sheet

**What's in here**
- `minimize` with a custom loss (piecewise heating/cooling model), `curve_fit`
- `minimize_scalar`, root finding with `brentq`
- Constrained optimisation: bounds + linear constraint (tiny dispatch problem), and the same with `linprog`
- Filling gaps: `np.interp`, `interp1d`, vs pandas `interpolate`
- `signal.detrend`, moving-average filters vs pandas rolling
- `spatial.distance.cdist`, `sparse` in one cell each

In [1]:
import numpy as np
import pandas as pd
from scipy import optimize, interpolate, signal

pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).sort_values("time").reset_index(drop=True)
# daily means keep the example small and reduce hour-of-day noise
daily = df.set_index("time").resample("D")[["consumption_mwh", "temp_c"]].mean()
temp = daily["temp_c"].values; cons = daily["consumption_mwh"].values
daily.head()

,consumption_mwh,temp_c
time,,
2022-01-01 00:00:00+00:00,30587.241667,1.753333
2022-01-02 00:00:00+00:00,30374.750000,3.155000
2022-01-03 00:00:00+00:00,31855.712500,3.412917
2022-01-04 00:00:00+00:00,32808.137500,1.701250
2022-01-05 00:00:00+00:00,33942.295833,-1.228333


## `minimize` with a custom loss

Model: consumption = a + b*HDD + c*CDD with HDD = max(15 - T, 0), CDD = max(T - 22, 0). Write the loss, hand it to `minimize`. Always check `res.success` and `res.message`, and try more than one starting point / method.

In [2]:
def model(params, T):
    a, b, c = params
    return a + b * np.clip(15 - T, 0, None) + c * np.clip(T - 22, 0, None)

def sse(params, T, y):
    return np.sum((y - model(params, T)) ** 2)

x0 = [cons.mean(), 0.0, 0.0]
res = optimize.minimize(sse, x0, args=(temp, cons), method="Nelder-Mead", options={"maxiter": 5000, "xatol": 1e-6, "fatol": 1e-6})
print(res.success, res.message, "n_eval:", res.nfev)
print("a, b, c =", res.x.round(1))
res2 = optimize.minimize(sse, x0, args=(temp, cons), method="L-BFGS-B")
print("L-BFGS-B:", res2.x.round(1), " same? ", np.allclose(res.x, res2.x, rtol=1e-2))

True Optimization terminated successfully. n_eval: 558
a, b, c = [27230.6   361.8 -1420.7]
L-BFGS-B: [27230.6   361.8 -1420.6]  same?  True


**Pitfall:** the loss is badly scaled (values ~1e11) and the parameters are on different scales (30000 vs 400). Gradient methods can stall. Scale the target (e.g. divide by 1000) or supply a gradient. Since this model is linear in (a, b, c), you should really use least squares directly - `minimize` is for when you cannot.

In [3]:
X = np.column_stack([np.ones_like(temp), np.clip(15 - temp, 0, None), np.clip(temp - 22, 0, None)])
beta, *_ = np.linalg.lstsq(X, cons, rcond=None)
print("lstsq a,b,c =", beta.round(1), "  (matches minimize:", np.allclose(beta, res.x, rtol=1e-2), ")")

lstsq a,b,c = [27230.6   361.8 -1420.7]   (matches minimize: True )


Robust alternative: swap the loss. A Huber loss shrinks the influence of outlier days - still one line with `minimize`.

In [4]:
def huber(params, T, y, delta=1500):
    r = y - model(params, T)
    return np.sum(np.where(np.abs(r) <= delta, 0.5 * r**2, delta * (np.abs(r) - 0.5 * delta)))

res_h = optimize.minimize(huber, res.x, args=(temp, cons), method="Nelder-Mead", options={"maxiter": 5000})
print("SSE  fit:", res.x.round(1))
print("Huber fit:", res_h.x.round(1))

SSE  fit: [27230.6   361.8 -1420.7]
Huber fit: [27302.6   362.  -1478.7]


## `curve_fit` - nonlinear least squares with standard errors

Give it `f(x, *params)`; it returns parameters and the covariance matrix. Uses Levenberg-Marquardt by default (bounds -> `trf`). Here we also let the heating threshold be a free parameter, which makes the model genuinely nonlinear.

In [5]:
def f(T, a, b, c, thr):
    return a + b * np.clip(thr - T, 0, None) + c * np.clip(T - 22, 0, None)

popt, pcov = optimize.curve_fit(f, temp, cons, p0=[30000, 300, 100, 15])
se = np.sqrt(np.diag(pcov))
pd.DataFrame({"param": ["a", "b (HDD slope)", "c (CDD slope)", "threshold"], "est": popt.round(2), "se": se.round(2), "t": (popt / se).round(1)})

,param,est,se,t
0,a,27259.95,76.06,358.4
1,b (HDD slope),365.96,10.27,35.6
2,c (CDD slope),-1444.30,442.32,-3.3
3,threshold,14.80,0.33,44.9


Note the CDD slope `c`: negative with a standard error of ~440. Very few *daily mean* temperatures exceed 22C, so that parameter is fitted on a handful of days and is essentially unidentified. A t-stat around 3 on a coefficient with the wrong sign is a red flag, not a discovery.

**Interview check:** "The threshold came out near 15 - is that a finding or did you assume it?" Here the data was generated with 15, so the fit recovers it. In real data the threshold and the slope are strongly correlated (check `pcov` off-diagonals) and the estimate is unstable - profile the loss over the threshold to see how flat it is.

In [6]:
thr_grid = np.arange(10, 20.5, 0.5)
prof = []
for thr in thr_grid:
    Xt = np.column_stack([np.ones_like(temp), np.clip(thr - temp, 0, None), np.clip(temp - 22, 0, None)])
    b_, *_ = np.linalg.lstsq(Xt, cons, rcond=None)
    prof.append(np.sum((cons - Xt @ b_) ** 2))
prof = np.array(prof) / min(prof)
print(pd.Series(prof.round(4), index=thr_grid).to_string())

10.0    1.2128
10.5    1.1608
11.0    1.1187
11.5    1.0862
12.0    1.0603
12.5    1.0394
13.0    1.0216
13.5    1.0094
14.0    1.0025
14.5    1.0000
15.0    1.0003
15.5    1.0056
16.0    1.0148
16.5    1.0257
17.0    1.0381
17.5    1.0521
18.0    1.0670
18.5    1.0790
19.0    1.0910
19.5    1.0998
20.0    1.1058


## `minimize_scalar` and root finding

One-dimensional problems: `minimize_scalar` (bounded or Brent), `brentq` for a root inside a bracket where the function changes sign. Example: at what temperature does the fitted model predict 32,000 MWh?

In [7]:
g = lambda thr: np.interp(thr, thr_grid, prof)                     # profile from above
r = optimize.minimize_scalar(g, bounds=(10, 20), method="bounded")
print("threshold minimising SSE:", round(r.x, 2))

target = 32000
h = lambda T: f(T, *popt) - target
print("sign change? h(-5)=", round(h(-5)), " h(15)=", round(h(15)))
T_star = optimize.brentq(h, -5, 15)
print(f"model hits {target} MWh at T = {T_star:.2f} C")

threshold minimising SSE: 14.5
sign change? h(-5)= 2505  h(15)= -4740
model hits 32000 MWh at T = 1.84 C


**Pitfall:** `brentq` needs `f(a)` and `f(b)` of opposite sign, otherwise `ValueError`. `fsolve` does not need a bracket but can return a non-root silently - always check the residual.

In [8]:
try:
    optimize.brentq(h, 20, 30)     # model is flat/rising here, no root
except ValueError as e:
    print("ValueError:", e)
# fsolve does not need a bracket, but it also does not tell you loudly when there is no root
no_root = lambda x: x**2 + 1
sol, info, ier, msg = optimize.fsolve(no_root, x0=3.0, full_output=True)
print("fsolve on x^2+1:", sol.round(3), " residual:", no_root(sol).round(3), " ier:", ier, "->", msg[:45])
print("always check the residual (and ier == 1), not just the returned x")

ValueError: f(a) and f(b) must have different signs
fsolve on x^2+1: [0.]  residual: [1.]  ier: 5 -> The iteration is not making good progress, as
always check the residual (and ier == 1), not just the returned x


## Constrained optimisation: a tiny dispatch problem

Meet demand D from three assets with marginal costs and capacities, minimising cost. With `minimize`: bounds for capacities, an equality constraint for the balance. This is linear so `linprog` is the right tool - but the `minimize` form generalises to nonlinear costs (e.g. quadratic fuel curves).

In [9]:
cost = np.array([30.0, 60.0, 120.0])       # EUR/MWh
cap = np.array([400.0, 300.0, 500.0])      # MWh max
D = 800.0

# nonlinear-capable form
obj = lambda q: cost @ q
cons_eq = {"type": "eq", "fun": lambda q: q.sum() - D}
res = optimize.minimize(obj, x0=np.full(3, D / 3), bounds=[(0, c) for c in cap], constraints=[cons_eq], method="SLSQP")
print("minimize/SLSQP:", res.x.round(1), " cost:", round(res.fun, 1), res.message)

# LP form: minimise c@x  s.t. A_eq x = b_eq, bounds
lp = optimize.linprog(c=cost, A_eq=np.ones((1, 3)), b_eq=[D], bounds=[(0, c) for c in cap], method="highs")
print("linprog/highs :", lp.x.round(1), " cost:", round(lp.fun, 1), lp.message)
print("shadow price of demand (EUR/MWh):", -lp.eqlin.marginals.round(1), " = marginal unit's cost")

minimize/SLSQP: [400. 300. 100.]  cost: 42000.0 Optimization terminated successfully
linprog/highs : [400. 300. 100.]  cost: 42000.0 Optimization terminated successfully. (HiGHS Status 7: Optimal)
shadow price of demand (EUR/MWh): [-120.]  = marginal unit's cost


Add a quadratic cost term (fuel efficiency falls with load) and SLSQP still works while `linprog` no longer applies. Also an inequality constraint: total emissions cap.

In [10]:
quad = np.array([0.02, 0.05, 0.01])
emis = np.array([0.9, 0.4, 0.2])            # tCO2/MWh
obj_q = lambda q: cost @ q + quad @ q**2
cons_list = [
    {"type": "eq",   "fun": lambda q: q.sum() - D},
    {"type": "ineq", "fun": lambda q: 350 - emis @ q},     # ineq means fun(q) >= 0
]
res = optimize.minimize(obj_q, x0=np.full(3, D / 3), bounds=[(0, c) for c in cap], constraints=cons_list, method="SLSQP")
print(res.x.round(1), " cost:", round(res.fun, 1), " emissions:", round(emis @ res.x, 1), res.message)

[185.7 300.  314.3]  cost: 67463.3  emissions: 350.0 Optimization terminated successfully


**Interview check:** "Your optimiser returned a solution - how do you know it is the optimum?" Check `success`, verify the constraints hold to tolerance, perturb the start point, and for convex problems compare with a specialised solver (LP -> `linprog`). For non-convex problems, there is no guarantee: use multi-start.

## Filling gaps: interpolation

`np.interp` is linear and fast (1-D, x must be increasing). `interp1d` gives other kinds. Pandas `interpolate(method="time")` handles irregular timestamps for you. Never interpolate the *target* you intend to forecast without saying so - you are inventing data.

In [11]:
s = df.set_index("time")["temp_c"].iloc[:240].copy()
gap = s.index[100:106]
s_gap = s.copy(); s_gap[gap] = np.nan

xs = np.arange(len(s_gap)); ok = ~s_gap.isna()
lin = np.interp(xs, xs[ok], s_gap.values[ok])
cub = interpolate.interp1d(xs[ok], s_gap.values[ok], kind="cubic")(xs)
pdi = s_gap.interpolate(method="time")

out = pd.DataFrame({"truth": s[gap], "np.interp": lin[100:106], "cubic": cub[100:106], "pandas_time": pdi[gap]}).round(2)
out

,truth,np.interp,cubic,pandas_time
time,,,,
2022-01-05 04:00:00+00:00,-4.36,-3.44,-3.43,-3.44
2022-01-05 05:00:00+00:00,-4.18,-2.88,-2.80,-2.88
2022-01-05 06:00:00+00:00,-2.81,-2.31,-2.14,-2.31
2022-01-05 07:00:00+00:00,-2.32,-1.74,-1.50,-1.74
2022-01-05 08:00:00+00:00,-1.07,-1.17,-0.91,-1.17
2022-01-05 09:00:00+00:00,-0.78,-0.61,-0.41,-0.61


**Pitfall:** `interp1d` raises on extrapolation unless `fill_value="extrapolate"`; `np.interp` silently clamps to the edge values. Both are wrong choices for a leading/trailing gap in a forecasting setting - use `ffill` (last known value) if anything.

In [12]:
fi = interpolate.interp1d([0, 1, 2], [10, 20, 30])
try:
    fi(3)
except ValueError as e:
    print("interp1d:", e)
print("np.interp clamps:", np.interp([3, -1], [0, 1, 2], [10, 20, 30]))
print("with extrapolate :", interpolate.interp1d([0, 1, 2], [10, 20, 30], fill_value="extrapolate")(3))

interp1d: A value (3.0) in x_new is above the interpolation range's maximum value (2).
np.interp clamps: [30. 10.]
with extrapolate : 40.0


## Detrending and smoothing

`signal.detrend` removes a linear (or constant) trend. A moving average via `np.convolve` is centred (uses future values!) - pandas `rolling().mean()` is trailing by default, which is what you want for features.

In [13]:
y = daily["consumption_mwh"].values
det = signal.detrend(y)                       # remove linear trend
slope = np.polyfit(np.arange(len(y)), y, 1)[0]
print(f"trend removed: {slope:.1f} MWh/day  -> {slope*365:.0f} MWh/year")

w = 7
centred = np.convolve(y, np.ones(w) / w, mode="same")               # uses t-3..t+3
trailing = pd.Series(y).rolling(w).mean().values                    # uses t-6..t
print(pd.DataFrame({"y": y[6:12], "centred_MA": centred[6:12], "trailing_MA": trailing[6:12]}).round(0))

trend removed: -1.7 MWh/day  -> -634 MWh/year
         y  centred_MA  trailing_MA
0  33829.0     32628.0      32469.0
1  30852.0     32668.0      32507.0
2  30189.0     32568.0      32480.0
3  32893.0     32514.0      32628.0
4  33088.0     32309.0      32668.0
5  33243.0     32303.0      32568.0


**Interview check:** "Your smoothed feature uses `np.convolve(..., mode='same')` - is that allowed?" Not for forecasting: the centred window includes 3 future days. Use a trailing window (`rolling`) and remember even that includes *today* unless you `shift(1)` first.

## Distances and sparse matrices (brief)

`cdist` gives all pairwise distances between two sets of rows - useful for nearest-neighbour analog forecasting (find historic days most similar to today). `scipy.sparse` matters when one-hot encoding many categories: sklearn's `OneHotEncoder` returns sparse by default.

In [14]:
from scipy.spatial.distance import cdist
from scipy import sparse

# daily profiles: 24-hour consumption vectors, find the 3 days most similar to the last day
prof = df.assign(date=df["time"].dt.date, hour=df["time"].dt.hour).pivot_table(index="date", columns="hour", values="consumption_mwh").dropna()
last = prof.iloc[[-1]].values
d = cdist(last, prof.iloc[:-1].values, metric="euclidean")[0]
print("closest analog days:", prof.index[:-1][np.argsort(d)[:3]].tolist())

# sparse one-hot of hour
oh = sparse.csr_matrix((np.ones(len(df)), (np.arange(len(df)), df["time"].dt.hour.values)), shape=(len(df), 24))
print(type(oh).__name__, oh.shape, "non-zeros:", oh.nnz, " dense would be", oh.shape[0]*oh.shape[1], "cells")
print("row sums all 1:", np.allclose(np.asarray(oh.sum(axis=1)).ravel(), 1))

closest analog days: [datetime.date(2023, 2, 12), datetime.date(2022, 10, 5), datetime.date(2022, 12, 10)]
csr_matrix (17520, 24) non-zeros: 17520  dense would be 420480 cells
row sums all 1: True


## Quick reference

| Task | Call |
|---|---|
| Minimise custom loss | `optimize.minimize(f, x0, args=(...), method="L-BFGS-B", bounds=...)` |
| Nonlinear LS with SEs | `optimize.curve_fit(f, x, y, p0=...)` -> `popt, pcov` |
| 1-D minimum | `optimize.minimize_scalar(f, bounds=(a, b), method="bounded")` |
| Root in bracket | `optimize.brentq(f, a, b)` |
| LP | `optimize.linprog(c, A_ub, b_ub, A_eq, b_eq, bounds, method="highs")` |
| Linear interp | `np.interp(x_new, x, y)` |
| Other interp | `interpolate.interp1d(x, y, kind="cubic", fill_value="extrapolate")` |
| Detrend | `signal.detrend(y)` |
| Pairwise distance | `spatial.distance.cdist(A, B)` |